# الدرس 07 (الجزء الثاني): بناء سلاسل الـ RAG باستخدام FAISS و LCEL
### Building Complete RAG Pipelines with Meta FAISS & LCEL (Lecture 33)

في هذا الدفتر نطبق معمارية الـ RAG المتكاملة باستخدام:
- مسترجع Meta FAISS المحفوظ محلياً.
- لغة تعبير لانج تشين (LCEL).
- نموذج Groq LLM فائق السرعة.
- الأنماط الثلاثة: البسيط (Simple)، اللحظي (Streaming)، والتفاعلي (Conversational).

### 1. استيراد المكتبات وتحميل متغيرات البيئة
**آلية العمل:**
نقوم بتحميل مفتاح `GROQ_API_KEY` من ملف `.env`، واستيراد مكونات لانج تشين الأساسية:
- كائن `FAISS` لتحميل الفهرس الذي بنيناه وحفظناه محلياً في الدرس السابق.
- كائنات `LCEL` ومشغلات الربط: `ChatPromptTemplate`, `MessagesPlaceholder`, `RunnablePassthrough`, `StrOutputParser`.
- نموذج `ChatGroq` فائق السرعة لإنتاج الإجابات في أجزاء من الثانية.

In [ ]:
# import standard libraries and load environment variables
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# resolve centralized faiss database path
data_dir = Path("../../data") if Path("../../data").exists() else Path("./data")
faiss_db_dir = str(data_dir / "databases" / "faiss_index")

# import langchain components
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage

print("Libraries imported successfully!")
print(f"Loading FAISS index from: {faiss_db_dir}")

### 2. تحميل فهرس FAISS المحفوظ وتجهيز المسترجع (Retriever)
**آلية العمل:**
نفتح فهرس `FAISS` المحفوظ على القرص عبر `load_local` مع تفعيل `allow_dangerous_deserialization=True`.
ثم نحوله فوراً إلى `Retriever` باستخدام ميثود `.as_retriever()` مع تحديد نوع البحث `similarity` واسترجاع أفضل 3 قطع مطابقة (`k=3`).

In [ ]:
# initialize embedding function matching the stored index
if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
else:
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# load persistent faiss index from disk
vector_store = FAISS.load_local(
    faiss_db_dir,
    embeddings,
    allow_dangerous_deserialization=True
)

# convert vector store into a retriever
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print(f"FAISS vector store loaded! Total vectors in index: {vector_store.index.ntotal}")
print(f"Retriever configured: top_k={retriever.search_kwargs['k']}")

### 3. تهيئة نموذج اللغة فائق السرعة (Groq LLM)
**آلية العمل:**
نقوم بتشغيل نموذج `ChatGroq` مع ضبط درجة الحرارة على قيمة منخفضة (`temperature=0.2`) لضمان الالتزام الصارم بالحقائق المذكورة في مستندات FAISS وتفادي الهلوسة.

In [ ]:
# initialize fast Groq LLM
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)
print("Groq LLM initialized successfully!")

### 4. دالة تنسيق المستندات والبيانات الوصفية (`format_docs`)
**آلية العمل:**
المسترجع يرجع قائمة من كائنات `Document` تحتوي على نصوص وبيانات وصفية (`source`, `topic`).
تقوم هذه الدالة بدمج القطع في نص واحد منظم يحمل رقم المستند ومصدره ليسهل على الـ LLM فهم المراجع وصياغة إجابة معزوة للمصدر.

In [ ]:
# helper function to format retrieved documents into structured context
def format_docs(docs):
    formatted_chunks = []
    for idx, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "Unknown")
        topic = doc.metadata.get("topic", "General")
        formatted_chunks.append(
            f"[Document {idx} | Source: {source} | Topic: {topic}]:\n{doc.page_content}"
        )
    return "\n\n".join(formatted_chunks)

# test retriever and formatting
sample_docs = retriever.invoke("What is machine learning?")
print(f"Retrieved {len(sample_docs)} sample chunks. Formatted preview:\n")
print(format_docs(sample_docs)[:220], "...")

### 5. السلسلة الأولى: مسار RAG البسيط المباشر (Simple RAG Chain)
**آلية العمل:**
نركب أول مسار متكامل باستخدام مشغل الربط (`|`) في LCEL:
1. نمرر السؤال بالتوازي: جزء يبحث في FAISS عبر `retriever | format_docs` ويملأ خانة `{context}`، وجزء يمرر السؤال كما هو عبر `RunnablePassthrough()` لخانة `{question}`.
2. يُحقن القاموس في الـ `prompt`.
3. يعالج `llm` المدخلات ويصوغ الرد.
4. يستخرج `StrOutputParser()` النص النهائي الصافي.

In [ ]:
# define context-grounded prompt
simple_template = """You are a concise AI assistant. Answer the question using ONLY the following context.
If the answer cannot be found in the context, say "I don't know based on the provided documents."

Context:
{context}

Question: {question}
Answer:"""
simple_prompt = ChatPromptTemplate.from_template(simple_template)

# construct simple lcel rag chain
simple_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
    | StrOutputParser()
)

# test simple rag chain
query_simple = "What is the difference between AI and machine learning?"
answer_simple = simple_rag_chain.invoke(query_simple)

print(f"❓ Question: {query_simple}\n")
print(f"💬 Simple RAG Answer:\n{answer_simple}")

### 6. السلسلة الثانية: مسار RAG مع البث المباشر (Streaming RAG Chain)
**آلية العمل:**
واحدة من أقوى مميزات LCEL مع Groq هي دعم البث التدفقي اللحظي (.stream()).
هنا لا ننتظر حتى تكتمل الإجابة، بل نطبع كل رمز (Token) فور خروجه من معالج Groq LPU، مما يمنح استجابة فورية فائقة السرعة.

In [ ]:
# streaming rag chain (same composition, consumed via .stream)
query_stream = "What is deep learning and how does it relate to neural networks?"
print(f"❓ Streaming Question: {query_stream}\n")
print("⚡ Live Streaming Output: ")

for chunk in simple_rag_chain.stream(query_stream):
    print(chunk, end="", flush=True)

print("\n\n[Streaming Finished Successfully!]")

### 7. السلسلة الثالثة: مسار RAG التفاعلي مع ذاكرة المحادثة (Conversational RAG Chain)
**آلية العمل:**
في المحادثات الحقيقية، يسأل المستخدم أسئلة تتابعية تحتوي على ضمائر مثل: *"وكيف يختلف عن البرمجة التقليدية؟"*.
نستخدم `MessagesPlaceholder("chat_history")` لحقن سجل الرسائل السابقة داخل الـ Prompt، مع استخدام `RunnablePassthrough.assign` لاستخراج وتنسيق السياق المناسب.

In [ ]:
# define conversational prompt template with chat history placeholder
conv_template = """You are a helpful AI assistant. Answer the question using the provided context and conversation history.
Context:
{context}
"""
conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", conv_template),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}")
])

# construct conversational lcel rag pipeline
conversational_rag_chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["question"]))
    )
    | conversational_prompt
    | llm
    | StrOutputParser()
)
print("Conversational RAG chain initialized successfully!")

### 8. اختبار المحادثة متعددة الجولات والتعامل مع الضمائر
**آلية العمل:**
- **الجولة الأولى:** نسأل عن موضوع عام: *"What is machine learning?"*، ثم نحفظ السؤال والجواب كـ `HumanMessage` و `AIMessage` داخل `chat_history`.
- **الجولة الثانية:** نسأل سؤالاً ناقصاً بالضمير: *"How is it different from traditional programming?"*.
- بفضل الذاكرة والسياق، يفهم النموذج فوراً أن المقصود بـ `it` هو `machine learning`، ويجيب بدقة كاملة!

In [ ]:
# initialize empty conversation memory
chat_history = []

# turn 1: initial question
q1 = "What is machine learning?"
a1 = conversational_rag_chain.invoke({"question": q1, "chat_history": chat_history})

print(f"👤 Human (Turn 1): {q1}\n")
print(f"🤖 Assistant (Turn 1):\n{a1}\n")

# update chat history
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])

# turn 2: follow-up question referencing 'it'
q2 = "How is it different from traditional programming?"
a2 = conversational_rag_chain.invoke({"question": q2, "chat_history": chat_history})

print("=" * 60)
print(f"👤 Human (Turn 2): {q2}\n")
print(f"🤖 Assistant (Turn 2 - Context-Aware):\n{a2}\n")

# update history
chat_history.extend([
    HumanMessage(content=q2),
    AIMessage(content=a2)
])
print(f"Total conversation turns stored: {len(chat_history) // 2} rounds ({len(chat_history)} messages)")

### 9. اختبار الاستعلام عبر قائمة أسئلة متعددة (Batch Testing Loop)
**آلية العمل:**
نختبر قوة وثبات مسار FAISS RAG بتمرير قائمة من الأسئلة المتنوعة في حلقة تكرارية (`for loop`) ومقارنة الإجابات المعتمدة على الفهرس.

In [ ]:
# list of diverse test questions
test_questions = [
    "What is natural language processing and what are its core tasks?",
    "What architectures are used in deep learning for visual and textual data?"
]

print("Running batch evaluation on test questions:\n")
for idx, query in enumerate(test_questions, start=1):
    print(f"--- Question [{idx}]: {query} ---")
    ans = simple_rag_chain.invoke(query)
    print(f"Answer:\n{ans}\n")

## 🎯 ملخص الدرس 07 (المحاضرة 33):
1. **FAISS كـ Retriever**: تحويل الفهرس السريع إلى مسترجع ذكي يتم دمجه في ثوانٍ داخل أي خط أنابيب LCEL.
2. **الأنماط الثلاثة**: قمنا بتطبيق الـ RAG البسيط، والـ RAG المتدفق (Streaming)، والـ RAG التفاعلي بالذاكرة (Conversational).
3. **التكامل بين C++ و Python**: أثبتت التجربة كيف يمكن دمج مكتبة بحث فائقة السرعة مكتوبة بـ C++ (FAISS) مع نماذج لغة سريعة بـ LPUs (Groq) بسلاسة مذهلة.